In [1]:
import pandas as pd


In [2]:
train = pd.read_csv('train.csv')

In [3]:
test = pd.read_csv('test.csv')

In [4]:
print('missing values')
print(train.isnull().sum())

missing values
Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64


In [5]:
train['RoadType'] = train['RoadType'].fillna('Unknown')
test['RoadType'] = test['RoadType'].fillna('Unknown')

In [6]:
train['Weather'] = train['Weather'].fillna('Unknown')
test['Weather'] = test['Weather'].fillna('Unknown')

In [7]:
train['Temperature'] = train.groupby('day')['Temperature'].transform(lambda x: x.fillna(x.median()))
test['Temperature'] = test.groupby('day')['Temperature'].transform(lambda x: x.fillna(x.median()))

In [8]:
print(train.isnull().sum())
print(test.isnull().sum())

Index            0
geohash          0
day              0
timestamp        0
demand           0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
dtype: int64
Index            0
geohash          0
day              0
timestamp        0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
dtype: int64


In [9]:
def feature_engineering(df):
  df[['Hour', 'Minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
  df = df.drop(columns=['timestamp'])
  df['LargeVehicles'] = df['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0})
  df['Landmarks'] = df['Landmarks'].map({'Yes': 1, 'No': 0})
  return df

In [10]:
train = feature_engineering(train)
test = feature_engineering(test)

In [11]:
train.head()

,Index,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,Hour,Minute
0,0,qp02z1,48,0.048804,Unknown,1,0,0,16.398024,Unknown,0,0
1,1,qp02zt,48,0.118507,Residential,3,1,1,31.104565,Sunny,0,0
2,2,qp08bj,48,0.027132,Residential,1,0,0,25.919267,Sunny,0,0
3,3,qp08gt,48,0.003272,Residential,1,0,0,16.398024,Rainy,0,0
4,4,qp02zq,48,0.010819,Residential,1,0,0,10.803667,Rainy,0,0


In [12]:
train.tail()

,Index,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,Hour,Minute
77294,77294,qp0d4n,49,0.067203,Residential,1,0,0,11.501664,Rainy,2,0
77295,77295,qp0d4q,49,0.022859,Residential,3,1,1,14.715254,Foggy,2,0
77296,77296,qp0d4w,49,0.141342,Residential,3,1,1,19.678860,Sunny,2,0
77297,77297,qp0dhw,49,0.087574,Residential,1,0,0,22.573958,Sunny,2,0
77298,77298,qp0djq,49,0.002944,Residential,3,1,1,1.322034,Snowy,2,0


In [13]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00


In [14]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

In [15]:
X = train.drop(columns=['demand', 'Index']) # We drop 'demand' and 'Index' from the training data
y = train['demand']

In [16]:
test_features = test.drop(columns=['Index'])

In [17]:
cat_cols = ['geohash', 'RoadType', 'Weather'] # Tell CatBoost which columns are text/categorical

In [18]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42) #Split the train data (80% for training, 20% for testing our accuracy internally)

In [19]:
model = CatBoostRegressor(
    iterations=500,        # Number of decision trees
    learning_rate=0.1,     # How fast it learns
    depth=8,               # How deep the trees go
    eval_metric='RMSE',    # Root Mean Square Error (standard hackathon metric)
    cat_features=cat_cols, # Passing our text columns
    random_seed=42,
    verbose=50             # Print an update every 50 steps
)

In [20]:

model.fit(X_train, y_train, eval_set=(X_val, y_val)) #Train the Model!

0:	learn: 0.1317388	test: 0.1315850	best: 0.1315850 (0)	total: 144ms	remaining: 1m 11s
50:	learn: 0.0456242	test: 0.0450702	best: 0.0450702 (50)	total: 4.63s	remaining: 40.8s
100:	learn: 0.0411051	test: 0.0413898	best: 0.0413898 (100)	total: 12.3s	remaining: 48.6s
150:	learn: 0.0386205	test: 0.0394027	best: 0.0394027 (150)	total: 17.1s	remaining: 39.5s
200:	learn: 0.0373431	test: 0.0386044	best: 0.0386044 (200)	total: 20.2s	remaining: 30.1s
250:	learn: 0.0361004	test: 0.0378981	best: 0.0378981 (250)	total: 23.2s	remaining: 23s
300:	learn: 0.0348983	test: 0.0370640	best: 0.0370635 (299)	total: 25.7s	remaining: 17s
350:	learn: 0.0337688	test: 0.0362980	best: 0.0362980 (350)	total: 28.2s	remaining: 12s
400:	learn: 0.0328236	test: 0.0358150	best: 0.0358150 (400)	total: 31.2s	remaining: 7.7s
450:	learn: 0.0322373	test: 0.0355728	best: 0.0355728 (450)	total: 35s	remaining: 3.81s
499:	learn: 0.0317546	test: 0.0353411	best: 0.0353390 (498)	total: 37.5s	remaining: 0us

bestTest = 0.03533899418


CatBoostRegressor(cat_features=['geohash', 'RoadType', 'Weather'], depth=8, eval_metric='RMSE', iterations=500, learning_rate=0.1, loss_function='RMSE', random_seed=42, verbose=50)

In [21]:
predictions = model.predict(test_features) #Predict the 'demand' for the future test dataset

In [22]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': predictions
}) #the final Submission of CSV

In [23]:
submission.to_csv('my_first_submission.csv', index=False)
print("\nSUCCESS! 'my_first_submission.csv' has been created.")


SUCCESS! 'my_first_submission.csv' has been created.


In [24]:
import pandas as pd

# Reload train and test dataframes and apply initial preprocessing
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Fill missing 'RoadType' values
train['RoadType'] = train['RoadType'].fillna('Unknown')
test['RoadType'] = test['RoadType'].fillna('Unknown')

# Fill missing 'Weather' values
train['Weather'] = train['Weather'].fillna('Unknown')
test['Weather'] = test['Weather'].fillna('Unknown')

# Fill missing 'Temperature' values with median grouped by day
train['Temperature'] = train.groupby('day')['Temperature'].transform(lambda x: x.fillna(x.median()))
test['Temperature'] = test.groupby('day')['Temperature'].transform(lambda x: x.fillna(x.median()))

# Define and apply initial feature engineering function
def feature_engineering(df):
  df[['Hour', 'Minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
  df = df.drop(columns=['timestamp'])
  df['LargeVehicles'] = df['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0})
  df['Landmarks'] = df['Landmarks'].map({'Yes': 1, 'No': 0})
  return df

train = feature_engineering(train)
test = feature_engineering(test)

# Define the advanced_features function
def advanced_features(df):
    df['is_rush_hour'] = df['Hour'].apply(lambda x: 1 if (8 <= x <= 11) or (18 <= x <= 21) else 0)
    df['is_night'] = df['Hour'].apply(lambda x: 1 if (x >= 23 or x <= 5) else 0)
    df['lane_capacity'] = df['NumberofLanes'] * (df['LargeVehicles'] + 1)
    return df

# Apply advanced features
train = advanced_features(train)
test = advanced_features(test)

geohash_avg = train.groupby('geohash')['demand'].mean().to_dict()

train['geohash_avg_demand'] = train['geohash'].map(geohash_avg)
test['geohash_avg_demand'] = test['geohash'].map(geohash_avg)

global_mean = train['demand'].mean()
test['geohash_avg_demand'] = test['geohash_avg_demand'].fillna(global_mean)

print("Advanced Features Successfully Added!")
print("New columns in train:", train.shape[1])

Advanced Features Successfully Added!
New columns in train: 16


In [25]:
import numpy as np
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor

def add_cyclic_features(df):
    # There are 24 hours in a day, 60 minutes in an hour
    df['hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24.0)
    df['minute_sin'] = np.sin(2 * np.pi * df['Minute'] / 60.0)
    df['minute_cos'] = np.cos(2 * np.pi * df['Minute'] / 60.0)
    return df

train = add_cyclic_features(train)
test = add_cyclic_features(test)

X = train.drop(columns=['demand', 'Index'])
y = train['demand']
test_features = test.drop(columns=['Index'])
cat_cols = ['geohash', 'RoadType', 'Weather']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_predictions = np.zeros(len(test_features)) # Empty array to hold our 5 predictions

print("Starting 5-Fold Cross Validation...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"--- Training Model {fold + 1} of 5 ---")
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=1500,           # Tripled the number of trees
        learning_rate=0.05,        # Slower learning = higher accuracy
        depth=8,
        eval_metric='RMSE',
        cat_features=cat_cols,
        random_seed=42,
        early_stopping_rounds=100, # Stops training early if the score stops improving
        verbose=250                # Print updates less often to keep the screen clean
    )

    model.fit(X_tr, y_tr, eval_set=(X_va, y_va))

    test_predictions += model.predict(test_features) / kf.n_splits


submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_predictions
})

submission.to_csv('submission_v2_kfold2.csv', index=False)
print("SUCCESS! submission_v2_kfold2.csv has been created")

Starting 5-Fold Cross Validation...

--- Training Model 1 of 5 ---
0:	learn: 0.1362129	test: 0.1362235	best: 0.1362235 (0)	total: 89.8ms	remaining: 2m 14s
250:	learn: 0.0342069	test: 0.0359717	best: 0.0359661 (248)	total: 17.7s	remaining: 1m 27s
500:	learn: 0.0312874	test: 0.0342229	best: 0.0342229 (500)	total: 35.2s	remaining: 1m 10s
750:	learn: 0.0296466	test: 0.0335103	best: 0.0335103 (750)	total: 51.8s	remaining: 51.6s
1000:	learn: 0.0283732	test: 0.0330953	best: 0.0330945 (999)	total: 1m 9s	remaining: 34.8s
1250:	learn: 0.0275152	test: 0.0328628	best: 0.0328625 (1249)	total: 1m 27s	remaining: 17.4s
1499:	learn: 0.0267055	test: 0.0326303	best: 0.0326280 (1491)	total: 1m 43s	remaining: 0us

bestTest = 0.03262801845
bestIteration = 1491

Shrink model to first 1492 iterations.
--- Training Model 2 of 5 ---
0:	learn: 0.1355064	test: 0.1389763	best: 0.1389763 (0)	total: 79.8ms	remaining: 1m 59s
250:	learn: 0.0346307	test: 0.0366123	best: 0.0366123 (250)	total: 17.5s	remaining: 1m 26s
50

In [26]:
!pip install lightgbm

In [27]:
# import lightgbm as lgb
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import KFold
# from catboost import CatBoostRegressor
# from lightgbm import early_stopping, log_evaluation

# if 'geohash_avg_demand' in train.columns:
#     train = train.drop(columns=['geohash_avg_demand'])
#     test = test.drop(columns=['geohash_avg_demand'])

# cat_cols = ['geohash', 'RoadType', 'Weather']
# for c in cat_cols:
#     train[c] = train[c].astype('category')
#     test[c] = test[c].astype('category')

# X = train.drop(columns=['demand', 'Index'])
# y = train['demand']
# test_features = test.drop(columns=['Index'])

# cb_test_predictions = np.zeros(len(test_features))
# lgb_test_predictions = np.zeros(len(test_features))
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# print("Starting the Ultimate Ensemble (CatBoost + LightGBM)...")

# for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#     print(f"\n--- Training Fold {fold + 1} of 5 ---")
#     X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
#     X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

#     print("Training CatBoost...")
#     cb_model = CatBoostRegressor(
#         iterations=1500, learning_rate=0.05, depth=8,
#         eval_metric='RMSE', cat_features=cat_cols,
#         random_seed=42, early_stopping_rounds=100, verbose=0 # Verbose=0 keeps the screen clean
#     )
#     cb_model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
#     cb_test_predictions += cb_model.predict(test_features) / kf.n_splits

#     print("Training LightGBM...")
#     lgb_model = lgb.LGBMRegressor(
#         n_estimators=1500, learning_rate=0.05, max_depth=8,
#         random_state=42, n_jobs=-1
#     )
#     lgb_model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_va, y_va)],
#         callbacks=[early_stopping(100, verbose=False), log_evaluation(0)]
#     )
#     lgb_test_predictions += lgb_model.predict(test_features) / kf.n_splits

# print("\nBlending predictions...")
# final_predictions = (cb_test_predictions * 0.5) + (lgb_test_predictions * 0.5)

# submission = pd.DataFrame({
#     'Index': test['Index'],
#     'demand': final_predictions
# })

# submission.to_csv('submission_v3_ensemble.csv', index=False)
# print("SUCCESS! 'submission_v3_ensemble.csv' has been created. Ready to submit!")

In [28]:
# import lightgbm as lgb
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import KFold
# from catboost import CatBoostRegressor
# from lightgbm import early_stopping, log_evaluation

# if 'geohash_avg_demand' in train.columns:
#     train = train.drop(columns=['geohash_avg_demand'])
#     test = test.drop(columns=['geohash_avg_demand'])

# train['geo_zone'] = train['geohash'].str[:5]       # The immediate neighborhood
# test['geo_zone'] = test['geohash'].str[:5]

# train['geo_district'] = train['geohash'].str[:4]   # The larger city district
# test['geo_district'] = test['geohash'].str[:4]

# train['Road_Weather'] = train['RoadType'].astype(str) + "_" + train['Weather'].astype(str)
# test['Road_Weather'] = test['RoadType'].astype(str) + "_" + test['Weather'].astype(str)

# cat_cols = ['geohash', 'RoadType', 'Weather', 'geo_zone', 'geo_district', 'Road_Weather']
# for c in cat_cols:
#     train[c] = train[c].astype('category')
#     test[c] = test[c].astype('category')

# X = train.drop(columns=['demand', 'Index'])
# y = train['demand']
# test_features = test.drop(columns=['Index'])

# cb_test_predictions = np.zeros(len(test_features))
# lgb_test_predictions = np.zeros(len(test_features))
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# print("Starting the Ultimate Ensemble with Spatial Features...")

# for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#     print(f"\n--- Training Fold {fold + 1} of 5 ---")
#     X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
#     X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

#     print("Training CatBoost...")
#     cb_model = CatBoostRegressor(
#         iterations=1500, learning_rate=0.05, depth=8,
#         eval_metric='RMSE', cat_features=cat_cols,
#         random_seed=42, early_stopping_rounds=100, verbose=0
#     )
#     cb_model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
#     cb_test_predictions += cb_model.predict(test_features) / kf.n_splits

#     print("Training LightGBM...")
#     lgb_model = lgb.LGBMRegressor(
#         n_estimators=1500, learning_rate=0.05, max_depth=8,
#         random_state=42, n_jobs=-1
#     )
#     lgb_model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_va, y_va)],
#         callbacks=[early_stopping(100, verbose=False), log_evaluation(0)]
#     )
#     lgb_test_predictions += lgb_model.predict(test_features) / kf.n_splits

# print("\nBlending predictions...")
# final_predictions = (cb_test_predictions * 0.5) + (lgb_test_predictions * 0.5)

# submission = pd.DataFrame({
#     'Index': test['Index'],
#     'demand': final_predictions
# })

# submission.to_csv('submission_v4_spatial_ensemble.csv', index=False)
# print("SUCCESS! 'submission_v4_spatial_ensemble.csv' has been created.")

In [29]:
# import lightgbm as lgb
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import KFold
# from catboost import CatBoostRegressor
# from lightgbm import early_stopping, log_evaluation

# if 'geohash_avg_demand' in train.columns:
#     train = train.drop(columns=['geohash_avg_demand'])
#     test = test.drop(columns=['geohash_avg_demand'])

# train['geo_zone'] = train['geohash'].str[:5]
# test['geo_zone'] = test['geohash'].str[:5]
# train['geo_district'] = train['geohash'].str[:4]
# test['geo_district'] = test['geohash'].str[:4]
# train['Road_Weather'] = train['RoadType'].astype(str) + "_" + train['Weather'].astype(str)
# test['Road_Weather'] = test['RoadType'].astype(str) + "_" + test['Weather'].astype(str)


# train['time_key'] = train['geohash'] + "_" + train['Hour'].astype(str) + "_" + train['Minute'].astype(str)
# test['time_key'] = test['geohash'] + "_" + test['Hour'].astype(str) + "_" + test['Minute'].astype(str)

# day_48_data = train[train['day'] == 48]
# lag_dict = dict(zip(day_48_data['time_key'], day_48_data['demand']))

# train['yesterday_demand'] = train['time_key'].map(lag_dict)
# test['yesterday_demand'] = test['time_key'].map(lag_dict)

# geohash_mean = train.groupby('geohash')['demand'].mean().to_dict()
# global_mean = train['demand'].mean()

# train['yesterday_demand'] = train['yesterday_demand'].fillna(train['geohash'].map(geohash_mean)).fillna(global_mean)
# test['yesterday_demand'] = test['yesterday_demand'].fillna(test['geohash'].map(geohash_mean)).fillna(global_mean)

# train = train.drop(columns=['time_key'])
# test = test.drop(columns=['time_key'])

# cat_cols = ['geohash', 'RoadType', 'Weather', 'geo_zone', 'geo_district', 'Road_Weather']
# for c in cat_cols:
#     train[c] = train[c].astype('category')
#     test[c] = test[c].astype('category')

# X = train.drop(columns=['demand', 'Index'])
# y = train['demand']
# test_features = test.drop(columns=['Index'])

# cb_test_predictions = np.zeros(len(test_features))
# lgb_test_predictions = np.zeros(len(test_features))
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# print("Starting the Ultimate Ensemble with 24-Hour Time Lags...")

# for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#     print(f"\n--- Training Fold {fold + 1} of 5 ---")
#     X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
#     X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]


#     print("Training CatBoost...")
#     cb_model = CatBoostRegressor(
#         iterations=1500, learning_rate=0.05, depth=8,
#         eval_metric='RMSE', cat_features=cat_cols,
#         random_seed=42, early_stopping_rounds=100, verbose=0
#     )
#     cb_model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
#     cb_test_predictions += cb_model.predict(test_features) / kf.n_splits


#     print("Training LightGBM...")
#     lgb_model = lgb.LGBMRegressor(
#         n_estimators=1500, learning_rate=0.05, max_depth=8,
#         random_state=42, n_jobs=-1
#     )
#     lgb_model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_va, y_va)],
#         callbacks=[early_stopping(100, verbose=False), log_evaluation(0)]
#     )
#     lgb_test_predictions += lgb_model.predict(test_features) / kf.n_splits

# print("\nBlending predictions...")
# final_predictions = (cb_test_predictions * 0.5) + (lgb_test_predictions * 0.5)

# submission = pd.DataFrame({
#     'Index': test['Index'],
#     'demand': final_predictions
# })

# submission.to_csv('submission_v5_timelag.csv', index=False)
# print("SUCCESS! 'submission_v5_timelag.csv' has been created. Go get that high score!")

In [30]:
# import lightgbm as lgb
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import KFold
# from catboost import CatBoostRegressor
# from lightgbm import early_stopping, log_evaluation

# cols_to_remove = ['geohash_avg_demand', 'yesterday_demand']
# for c in cols_to_remove:
#     if c in train.columns: train = train.drop(columns=[c])
#     if c in test.columns: test = test.drop(columns=[c])

# train['is_rush_hour'] = train['Hour'].apply(lambda x: 1 if (8 <= x <= 11) or (18 <= x <= 21) else 0)
# test['is_rush_hour'] = test['Hour'].apply(lambda x: 1 if (8 <= x <= 11) or (18 <= x <= 21) else 0)

# train['is_night'] = train['Hour'].apply(lambda x: 1 if (x >= 23 or x <= 5) else 0)
# test['is_night'] = test['Hour'].apply(lambda x: 1 if (x >= 23 or x <= 5) else 0)

# train['lane_capacity'] = train['NumberofLanes'] * (train['LargeVehicles'] + 1)
# test['lane_capacity'] = test['NumberofLanes'] * (test['LargeVehicles'] + 1)

# train['geo_zone'] = train['geohash'].str[:5]
# test['geo_zone'] = test['geohash'].str[:5]

# train['geo_district'] = train['geohash'].str[:4]
# test['geo_district'] = test['geohash'].str[:4]

# train['Road_Weather'] = train['RoadType'].astype(str) + "_" + train['Weather'].astype(str)
# test['Road_Weather'] = test['RoadType'].astype(str) + "_" + test['Weather'].astype(str)

# cat_cols = ['geohash', 'RoadType', 'Weather', 'geo_zone', 'geo_district', 'Road_Weather']
# for c in cat_cols:
#     train[c] = train[c].astype('category')
#     test[c] = test[c].astype('category')

# X = train.drop(columns=['demand', 'Index'])
# y = train['demand']
# test_features = test.drop(columns=['Index'])

# cb_test_predictions = np.zeros(len(test_features))
# lgb_test_predictions = np.zeros(len(test_features))
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# print("Starting the Ultimate Combined Ensemble...")

# for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#     print(f"\n--- Training Fold {fold + 1} of 5 ---")
#     X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
#     X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

#     cb_model = CatBoostRegressor(
#         iterations=2000,
#         learning_rate=0.04,
#         depth=8,
#         eval_metric='RMSE', cat_features=cat_cols,
#         random_seed=42, early_stopping_rounds=150, verbose=0
#     )
#     cb_model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
#     cb_test_predictions += cb_model.predict(test_features) / kf.n_splits


#     lgb_model = lgb.LGBMRegressor(
#         n_estimators=2000,
#         learning_rate=0.04,
#         max_depth=8,
#         random_state=42, n_jobs=-1
#     )
#     lgb_model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_va, y_va)],
#         callbacks=[early_stopping(150, verbose=False), log_evaluation(0)]
#     )
#     lgb_test_predictions += lgb_model.predict(test_features) / kf.n_splits

# print("\nBlending predictions...")
# final_predictions = (cb_test_predictions * 0.5) + (lgb_test_predictions * 0.5)

# submission = pd.DataFrame({
#     'Index': test['Index'],
#     'demand': final_predictions
# })

# submission.to_csv('submission_v6_master.csv', index=False)
# print("SUCCESS! 'submission_v6_master.csv' has been created.")

In [31]:
import lightgbm as lgb
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor
from lightgbm import early_stopping, log_evaluation

cols_to_remove = ['geohash_avg_demand', 'yesterday_demand', 'is_rush_hour', 'is_night', 'lane_capacity']
for c in cols_to_remove:
    if c in train.columns: train = train.drop(columns=[c])
    if c in test.columns: test = test.drop(columns=[c])

train['geo_zone'] = train['geohash'].str[:5]
test['geo_zone'] = test['geohash'].str[:5]

train['geo_district'] = train['geohash'].str[:4]
test['geo_district'] = test['geohash'].str[:4]

train['Road_Weather'] = train['RoadType'].astype(str) + "_" + train['Weather'].astype(str)
test['Road_Weather'] = test['RoadType'].astype(str) + "_" + test['Weather'].astype(str)

cat_cols = ['geohash', 'RoadType', 'Weather', 'geo_zone', 'geo_district', 'Road_Weather']
for c in cat_cols:
    train[c] = train[c].astype('category')
    test[c] = test[c].astype('category')

X = train.drop(columns=['demand', 'Index'])
y = train['demand']
test_features = test.drop(columns=['Index'])

cb_test_predictions = np.zeros(len(test_features))
lgb_test_predictions = np.zeros(len(test_features))
xgb_test_predictions = np.zeros(len(test_features))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Starting the Holy Trinity Ensemble (CatBoost + LightGBM + XGBoost)...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1} of 5 ---")
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    print("Training CatBoost...")
    cb_model = CatBoostRegressor(
        iterations=1500, learning_rate=0.05, depth=8,
        eval_metric='RMSE', cat_features=cat_cols,
        random_seed=42, early_stopping_rounds=100, verbose=0
    )
    cb_model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    cb_test_predictions += cb_model.predict(test_features) / kf.n_splits

    print("Training LightGBM...")
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.05, max_depth=8,
        random_state=42, n_jobs=-1
    )
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[early_stopping(100, verbose=False), log_evaluation(0)]
    )
    lgb_test_predictions += lgb_model.predict(test_features) / kf.n_splits

    print("Training XGBoost...")
    xgb_model = xgb.XGBRegressor(
        n_estimators=1500, learning_rate=0.05, max_depth=8,
        enable_categorical=True, tree_method='hist',
        random_state=42, early_stopping_rounds=100, n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    xgb_test_predictions += xgb_model.predict(test_features) / kf.n_splits

print("\nBlending predictions...")
final_predictions = (cb_test_predictions * 0.334) + (lgb_test_predictions * 0.333) + (xgb_test_predictions * 0.333)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_predictions
})

submission.to_csv('submission_v7_trinity.csv', index=False)
print("SUCCESS! 'submission_v7_trinity.csv' has been created.")

Starting the Holy Trinity Ensemble (CatBoost + LightGBM + XGBoost)...

--- Training Fold 1 of 5 ---
Training CatBoost...
Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040012 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1504
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 17
[LightGBM] [Info] Start training from score 0.093784
Training XGBoost...

--- Training Fold 2 of 5 ---
Training CatBoost...
Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For c

In [32]:
# # 1. Force negative predictions to be exactly 0 (Traffic can't be negative!)
# submission['demand'] = submission['demand'].clip(lower=0)

# # 2. Force the 'Index' column to be an integer (HackerEarth rejects decimals here)
# submission['Index'] = submission['Index'].astype(int)

# # 3. Fill any accidental blank predictions (NaNs) with the average demand
# submission['demand'] = submission['demand'].fillna(submission['demand'].mean())

# # Check our work
# print("Number of negative predictions left:", (submission['demand'] < 0).sum())
# print("Number of missing values left:", submission['demand'].isnull().sum())

# # 4. Save the repaired file
# submission.to_csv('submission_v7_FIXED.csv', index=False)
# print("\nSUCCESS! Download 'submission_v7_FIXED.csv' and upload it to HackerEarth.")

Number of negative predictions left: 0
Number of missing values left: 0

SUCCESS! Download 'submission_v7_FIXED.csv' and upload it to HackerEarth.


In [33]:
import lightgbm as lgb
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor
from lightgbm import early_stopping, log_evaluation

# 1. CLEAN UP (Ensure no failed experiment columns remain)
cols_to_remove = ['geohash_avg_demand', 'yesterday_demand', 'is_rush_hour', 'is_night', 'lane_capacity']
for c in cols_to_remove:
    if c in train.columns: train = train.drop(columns=[c])
    if c in test.columns: test = test.drop(columns=[c])

# 2. THE CHAMPION SPATIAL FEATURES
train['geo_zone'] = train['geohash'].str[:5]
test['geo_zone'] = test['geohash'].str[:5]

train['geo_district'] = train['geohash'].str[:4]
test['geo_district'] = test['geohash'].str[:4]

train['Road_Weather'] = train['RoadType'].astype(str) + "_" + train['Weather'].astype(str)
test['Road_Weather'] = test['RoadType'].astype(str) + "_" + test['Weather'].astype(str)

# 3. THE MISSING PIECE: CYCLIC TIME (Teaching the AI that time is a loop)
train['hour_sin'] = np.sin(2 * np.pi * train['Hour'] / 24.0)
train['hour_cos'] = np.cos(2 * np.pi * train['Hour'] / 24.0)
train['minute_sin'] = np.sin(2 * np.pi * train['Minute'] / 60.0)
train['minute_cos'] = np.cos(2 * np.pi * train['Minute'] / 60.0)

test['hour_sin'] = np.sin(2 * np.pi * test['Hour'] / 24.0)
test['hour_cos'] = np.cos(2 * np.pi * test['Hour'] / 24.0)
test['minute_sin'] = np.sin(2 * np.pi * test['Minute'] / 60.0)
test['minute_cos'] = np.cos(2 * np.pi * test['Minute'] / 60.0)

# 4. PREPARE CATEGORICALS
cat_cols = ['geohash', 'RoadType', 'Weather', 'geo_zone', 'geo_district', 'Road_Weather']
for c in cat_cols:
    train[c] = train[c].astype('category')
    test[c] = test[c].astype('category')

# 5. SETUP FEATURES AND TARGET
X = train.drop(columns=['demand', 'Index'])
y = train['demand']
test_features = test.drop(columns=['Index'])

# 6. PREPARE TO STORE PREDICTIONS
cb_test_predictions = np.zeros(len(test_features))
lgb_test_predictions = np.zeros(len(test_features))
xgb_test_predictions = np.zeros(len(test_features))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Starting the V8 Final Boss (Trinity + Spatial + Cyclic Time)...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1} of 5 ---")
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    # --- MODEL A: CATBOOST ---
    cb_model = CatBoostRegressor(
        iterations=1500, learning_rate=0.05, depth=8,
        eval_metric='RMSE', cat_features=cat_cols,
        random_seed=42, early_stopping_rounds=100, verbose=0
    )
    cb_model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    cb_test_predictions += cb_model.predict(test_features) / kf.n_splits

    # --- MODEL B: LIGHTGBM ---
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.05, max_depth=8,
        random_state=42, n_jobs=-1
    )
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[early_stopping(100, verbose=False), log_evaluation(0)]
    )
    lgb_test_predictions += lgb_model.predict(test_features) / kf.n_splits

    # --- MODEL C: XGBOOST ---
    xgb_model = xgb.XGBRegressor(
        n_estimators=1500, learning_rate=0.05, max_depth=8,
        enable_categorical=True, tree_method='hist',
        random_state=42, early_stopping_rounds=100, n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    xgb_test_predictions += xgb_model.predict(test_features) / kf.n_splits

# 7. THE TRINITY BLEND
print("\nBlending predictions...")
final_predictions = (cb_test_predictions * 0.334) + (lgb_test_predictions * 0.333) + (xgb_test_predictions * 0.333)

# 8. SAFE-SAVE FINAL SUBMISSION (Prevents Leaderboard 0s!)
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_predictions
})

# Automatic Safety Fixes
submission['demand'] = submission['demand'].clip(lower=0)
submission['Index'] = submission['Index'].astype(int)
submission['demand'] = submission['demand'].fillna(submission['demand'].mean())

submission.to_csv('submission_v8_final_boss.csv', index=False)
print("SUCCESS! 'submission_v8_final_boss.csv' has been created. Ready to conquer!")

Starting the V8 Final Boss (Trinity + Spatial + Cyclic Time)...

--- Training Fold 1 of 5 ---
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005459 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1504
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 17
[LightGBM] [Info] Start training from score 0.093784

--- Training Fold 2 of 5 ---
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGB